___
# <font color= #003366> **Primer Examen Parcial** </font>
- <Strong> Integrantes: </Strong>  <font color="blue">`Clara Paola Aguilar Casillas, Roi Jared Flores Garza Stone, Mónica Ibarra Herrera, Diana Denise Valdivia Vargas` </font>
- <Strong> Materia: </Strong>  <font color="blue">`Modelos No Lineales Para Pronósticos` </font>

<div style="display: flex; align-items: center;">
    <div style="flex: 1;">
        <img src="https://oci02.img.iteso.mx/Identidades-De-Instancia/ITESO/Logos%20ITESO/Logo-ITESO-Principal.jpg" width="300">
    </div>
</div>

___

## <font color= #003366> **Objetivo** </font>

El objetivo del examen es desarrollar un modelo SARIMAX usando información obtenida mediante la API del Banco de México, con el fin de analizar y predecir el comportamiento del tipo de cambio de peso mexicano frente al dólar estadounidense.

Se busca identificar la estructura de la serie, evaluar su estacionariedad, estimar los parámetros óptimos del modelo y generar pronósticos diarios para el periodo del 5 al 13 de marzo de 2026.

Finalmente, se evaluará qué tan bien el puede capturar el modelo la dinámica del tipo de cambio.

## <font color= #003366> **Descripción de la Serie de Tiempo** </font>

La serie de tiempo que se usará en el examen corresponde al Tipo de Cambio FIX (peso mexicano – dólar estadounidense) publicado por el Banco de México.

Las características principales de la serie son:
- Nombre: Tipo de cambio FIX (MXN/USD)
- Fuente: API del Banco de México (SIE)
- Clave de serie: SF43718
- Frecuencia: Diaria
- Descripción: Indicador financiero publicado por Banxico que refleja el tipo de cambio peso mexicano - dólar estadounidense

## <font color= #003366> **Librerías** </font>

In [1]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from plotly.subplots import make_subplots
from statsmodels.tsa.stattools import adfuller, acf, pacf
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"

## <font color= #003366> **Conectar con la API** </font>

In [2]:
token = "c08c91a75598ee1e94c2081fb597314804b58642f10913c1e53f81c258ba5f17"

# Serie FIX
serie = "SF63528"

url = f"https://www.banxico.org.mx/SieAPIRest/service/v1/series/{serie}/datos"

headers = {
    "Bmx-Token": token
}

response = requests.get(url, headers=headers)

data = response.json()

In [3]:
datos = data['bmx']['series'][0]['datos']

df = pd.DataFrame(datos)
df.head()

,fecha,dato
0,19/04/1954,0.01250
1,20/04/1954,0.01250
2,21/04/1954,0.01250
3,22/04/1954,0.01250
4,23/04/1954,0.01250


In [4]:
df.columns = ['fecha', 'tipo_cambio']

df['fecha'] = pd.to_datetime(df['fecha'], format='%d/%m/%Y')

df['tipo_cambio'] = pd.to_numeric(df['tipo_cambio'], errors='coerce')

df = df.sort_values('fecha')

df.set_index('fecha', inplace=True)
df.head()

,tipo_cambio
fecha,
1954-04-19,0.0125
1954-04-20,0.0125
1954-04-21,0.0125
1954-04-22,0.0125
1954-04-23,0.0125


## <font color= #003366> **Visualización de la serie de tiempo** </font>

In [5]:
df_plot = df.reset_index()

fig = px.line(
    df_plot,
    x="fecha",
    y="tipo_cambio",
    title="Tipo de Cambio FIX MXN/USD"
)

fig.show()

## <font color= #003366> **Pruebas de Estacionareidad** </font>

In [6]:
def check_stationarity(series, title="Serie Original"):
    result = adfuller(series.dropna())
    print(f'ADF Test: {title}')
    print(f'Estadístico ADF: {result[0]:.4f}')
    print(f'p-value: {result[1]:.4f}')
    is_stationary = result[1] < 0.05
    print(f"¿Es estacionaria? {'SÍ' if is_stationary else 'NO'}\n")
    return is_stationary

# 1. Revisamos la serie original
check_stationarity(df['tipo_cambio'], "Tipo de Cambio Original")

# 2. Aplicamos Primera Diferencia (d=1)
df['diff_1'] = df['tipo_cambio'].diff()

# 3. Revisamos la serie diferenciada
check_stationarity(df['diff_1'], "Primera Diferencia (d=1)")

# Creamos una figura con 2 columnas (Subplots)
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Serie Original (No Estacionaria)", "Serie Diferenciada (Estacionaria d=1)")
)

# Gráfico 1: Serie Original
fig.add_trace(
    go.Scatter(x=df['tipo_cambio'].index, y=df['tipo_cambio'], name='Original'),
    row=1, col=1
)

# Gráfico 2: Serie Diferenciada
fig.add_trace(
    go.Scatter(x=df['diff_1'].index, y=df['diff_1'], name='Diferenciada'),
    row=1, col=2
)

# Ajustes de diseño
fig.update_layout(
    title_text="Comparativa: Efecto de la Diferenciación",
    showlegend=False, # Ocultamos leyenda
    height=500
)

fig.show()

ADF Test: Tipo de Cambio Original
Estadístico ADF: -0.1244
p-value: 0.9469
¿Es estacionaria? NO

ADF Test: Primera Diferencia (d=1)
Estadístico ADF: -20.9975
p-value: 0.0000
¿Es estacionaria? SÍ



Con la prueba de Estacionareidad podemos ver que la serie original **no** es estacionaria.

Se realizó la primera defirenciación y al volver a realizar la prueba podemos visualizar que la serie ya es estacionaria

## <font color= #003366> **Gráficas de ACF y PACF** </font>

In [7]:
ts_analysis = df['diff_1'].dropna()

# Parámetros
lags = 30  # 30 días
alpha = 0.05 # Nivel de significancia del 95%

# Cálculo de valores ACF y PACF
acf_vals = acf(ts_analysis, nlags=lags, alpha=alpha)[0][1:]
pacf_vals = pacf(ts_analysis, nlags=lags, alpha=alpha)[0][1:]

# Colocamos manualmente el intervalo de confianza para plotly
n = len(ts_analysis)
conf_interval = 1.96 / np.sqrt(n)


fig = make_subplots(rows=2, cols=1,
                    subplot_titles=("Función de Autocorrelación (ACF) - Determina MA(q)",
                                    "Autocorrelación Parcial (PACF) - Determina AR(p)"),
                    vertical_spacing=0.15)

# ACF

fig.add_trace(go.Bar(
    x=list(range(1, lags+1)), y=acf_vals,
    name='ACF', marker_color='rgb(31, 119, 180)', showlegend=False
), row=1, col=1)

# Intervalos de confianza (Sombreado)
fig.add_shape(type="rect",
    x0=0.5, y0=-conf_interval, x1=lags+0.5, y1=conf_interval,
    line=dict(color="rgba(0,0,0,0)"), fillcolor="rgba(0,0,0,0.1)",
    row=1, col=1
)
# Líneas límite
fig.add_hline(y=conf_interval, line_dash="dash", line_color="gray", row=1, col=1)
fig.add_hline(y=-conf_interval, line_dash="dash", line_color="gray", row=1, col=1)

# PACF

fig.add_trace(go.Bar(
    x=list(range(1, lags+1)), y=pacf_vals,
    name='PACF', marker_color='rgb(255, 127, 14)', showlegend=False
), row=2, col=1)

# Intervalos de confianza
fig.add_shape(type="rect",
    x0=0.5, y0=-conf_interval, x1=lags+0.5, y1=conf_interval,
    line=dict(color="rgba(0,0,0,0)"), fillcolor="rgba(0,0,0,0.1)",
    row=2, col=1
)
# Líneas límite
fig.add_hline(y=conf_interval, line_dash="dash", line_color="gray", row=2, col=1)
fig.add_hline(y=-conf_interval, line_dash="dash", line_color="gray", row=2, col=1)


fig.update_layout(
    title='<b>Diagnóstico de Estructura: ACF y PACF</b><br><sup>Serie Diferenciada</sup>',
    template='plotly_white',
    height=700,
    bargap=0.8 # Barras delgadas estilo lollipop
)

# Resaltar lags estacionales (7, 14, 21, 28) con líneas verticales rojas tenues
for i in [7, 14, 21, 28]:
    fig.add_vline(x=i, line_width=1, line_dash="dot", line_color="red", opacity=0.5)

fig.show()